In [1]:
import sys, os
path = os.path.abspath('../src')
sys.path.insert(0, path)

from gedih3.gedidriver import soc_file_tree, gedi_file_glob
from gedih3.utils import parallel_glob

In [6]:
f = '/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/soc'
f = '/gpfs/data1/vclgp/data/iss_gedi/soc'

glob_kwargs = {'version':2}
glob_pattern = gedi_file_glob(**glob_kwargs) if glob_kwargs else 'GEDI*.h5'
file_list = parallel_glob(f, glob_pattern, engine='threads', show_progress=True)

QUEUEING TASKS | :   0%|          | 0/1853 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1853 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1853 [00:00<?, ?it/s]

In [ ]:
import re
import numpy as np
from earthaccess.store import EarthAccessFile
from gedih3.gedidriver import GEDIFile, soc_prod_from_file
import pqdm.threads as pqdm

file_struct = f
to_list = True
glob_kwargs = {'version':2}

# def soc_file_tree(file_struct, to_list=False, glob_kwargs=None):
direct_access = False
if isinstance(file_struct, str) and os.path.isdir(file_struct):
    glob_pattern = gedi_file_glob(**glob_kwargs) if glob_kwargs else 'GEDI*.h5'        
    file_list = parallel_glob(file_struct, glob_pattern, show_progress=True, engine='threads')
elif (direct_access := isinstance(file_struct[0], EarthAccessFile)):
    file_list = [i.path for i in file_struct]
elif isinstance(file_struct[0], str):
    file_list = file_struct
else:
    raise ValueError("file_struct must be a directory or a list of file paths (or s3 links)")

def _parse_glob_pattern(f):
    return GEDIFile.from_filename(f).get_glob_pattern(product=None, level=None, ppds=None)


QUEUEING TASKS | :   0%|          | 0/1853 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1853 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1853 [00:00<?, ?it/s]

In [22]:
len(np.unique(file_globs))

91672

In [ ]:
from pqdm.threads import pqdm

file_globs = np.array(pqdm(file_list, _parse_glob_pattern, n_jobs=1+os.cpu_count()//2, disable=False))
file_array = np.array(file_struct) if direct_access else np.array(file_list)

soc_tree = {}
for i,f in enumerate(np.unique(file_globs)):
    print(f"Processing {i+1} of {len(np.unique(file_globs))}: {f}", end='\r')
    orb_track = re.sub(r'^GEDI\*_.*_(O\d{5}_\d{2}_T\d{5}).*\.h5$', r'\1', f)
    f_prods = file_array[file_globs == f]
    f_obj = {soc_prod_from_file(fp):fp for fp in f_prods.tolist()}
    soc_tree[orb_track] = dict(sorted(f_obj.items()))

# if to_list:
#     soc_tree = list(soc_tree.values())

# return soc_tree


QUEUEING TASKS | :   0%|          | 0/435151 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/435151 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/435151 [00:00<?, ?it/s]

Processing 1 of 91672: GEDI*_*_*_O01753_01_T01683_*_*_*_V**.h5
Processing 2 of 91672: GEDI*_*_*_O01753_02_T01683_*_*_*_V**.h5
Processing 3 of 91672: GEDI*_*_*_O01753_03_T01683_*_*_*_V**.h5
Processing 4 of 91672: GEDI*_*_*_O01753_04_T01683_*_*_*_V**.h5
Processing 5 of 91672: GEDI*_*_*_O01754_01_T02954_*_*_*_V**.h5
Processing 6 of 91672: GEDI*_*_*_O01754_02_T02954_*_*_*_V**.h5
Processing 7 of 91672: GEDI*_*_*_O01754_03_T02954_*_*_*_V**.h5
Processing 8 of 91672: GEDI*_*_*_O01754_04_T02954_*_*_*_V**.h5
Processing 9 of 91672: GEDI*_*_*_O01755_01_T02955_*_*_*_V**.h5
Processing 10 of 91672: GEDI*_*_*_O01755_02_T02955_*_*_*_V**.h5
Processing 11 of 91672: GEDI*_*_*_O01755_03_T02955_*_*_*_V**.h5
Processing 12 of 91672: GEDI*_*_*_O01755_04_T02955_*_*_*_V**.h5
Processing 13 of 91672: GEDI*_*_*_O01756_01_T00110_*_*_*_V**.h5
Processing 14 of 91672: GEDI*_*_*_O01756_02_T00110_*_*_*_V**.h5
Processing 15 of 91672: GEDI*_*_*_O01756_03_T00110_*_*_*_V**.h5
Processing 16 of 91672: GEDI*_*_*_O01756_04_T0011

KeyboardInterrupt: 

In [19]:
file_globs

array(['GEDI*_*_*_O05538_02_T02416_*_*_*_V**.h5',
       'GEDI*_*_*_O05541_04_T05112_*_*_*_V**.h5',
       'GEDI*_*_*_O05538_04_T02416_*_*_*_V**.h5', ...,
       'GEDI*_*_*_O24083_01_T08871_*_*_*_V**.h5',
       'GEDI*_*_*_O24077_02_T06325_*_*_*_V**.h5',
       'GEDI*_*_*_O24074_03_T09321_*_*_*_V**.h5'],
      shape=(435151,), dtype='<U39')

In [9]:
file_list = soc_file_tree(f, to_list=True, glob_kwargs=glob_kwargs)

TypeError: 'module' object is not callable

In [ ]:
import dask_geopandas
hexes = ['83804cfffffffff', '83804efffffffff']
d = "/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/h3"
ddf = dask_geopandas.read_parquet(d, gather_spatial_partitions=False, filters=[('h3_03', 'in', hexes)])
ddf

,shot_number,elev_lowestmode_l2a,lat_lowestmode_l2a,quality_flag_l2a,lon_lowestmode_l2a,delta_time_l2a,rh_000_l2a,rh_001_l2a,rh_002_l2a,rh_003_l2a,rh_004_l2a,rh_005_l2a,rh_006_l2a,rh_007_l2a,rh_008_l2a,rh_009_l2a,rh_010_l2a,rh_011_l2a,rh_012_l2a,rh_013_l2a,rh_014_l2a,rh_015_l2a,rh_016_l2a,rh_017_l2a,rh_018_l2a,rh_019_l2a,rh_020_l2a,rh_021_l2a,rh_022_l2a,rh_023_l2a,rh_024_l2a,rh_025_l2a,rh_026_l2a,rh_027_l2a,rh_028_l2a,rh_029_l2a,rh_030_l2a,rh_031_l2a,rh_032_l2a,rh_033_l2a,rh_034_l2a,rh_035_l2a,rh_036_l2a,rh_037_l2a,rh_038_l2a,rh_039_l2a,rh_040_l2a,rh_041_l2a,rh_042_l2a,rh_043_l2a,rh_044_l2a,rh_045_l2a,rh_046_l2a,rh_047_l2a,rh_048_l2a,rh_049_l2a,rh_050_l2a,rh_051_l2a,rh_052_l2a,rh_053_l2a,rh_054_l2a,rh_055_l2a,rh_056_l2a,rh_057_l2a,rh_058_l2a,rh_059_l2a,rh_060_l2a,rh_061_l2a,rh_062_l2a,rh_063_l2a,rh_064_l2a,rh_065_l2a,rh_066_l2a,rh_067_l2a,rh_068_l2a,rh_069_l2a,rh_070_l2a,rh_071_l2a,rh_072_l2a,rh_073_l2a,rh_074_l2a,rh_075_l2a,rh_076_l2a,rh_077_l2a,rh_078_l2a,rh_079_l2a,rh_080_l2a,rh_081_l2a,rh_082_l2a,rh_083_l2a,rh_084_l2a,rh_085_l2a,rh_086_l2a,rh_087_l2a,rh_088_l2a,rh_089_l2a,rh_090_l2a,rh_091_l2a,rh_092_l2a,rh_093_l2a,rh_094_l2a,rh_095_l2a,rh_096_l2a,rh_097_l2a,rh_098_l2a,rh_099_l2a,rh_100_l2a,root_beam_l2a,root_file_l2a,agbd_l4a,root_beam_l4a,root_file_l4a,wsci_l4c,root_beam_l4c,root_file_l4c,datetime,geometry,h3_03,year
npartitions=14,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,uint64,float32,float64,uint8,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,string,string,float32,string,string,float32,string,string,datetime64[ns],geometry,category[known],category[known]
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..